# Federated Learning with Flower

We turn the LSTM into a federated model, where each WESAD subject becomes a separate Flower client and only model weights circulate through the network, raw data remains on the client.

The strategy used for aggregating: FedAvg, sample-weighted averaging of client weight updates. Useful for WESAD because the recorded subjects produced similar amounts of data and roughly the same set of classes.

In [ ]:
import sys, pathlib, importlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

# Force-reload so the kernel picks up any config/code changes without a restart
import src.config, src.model, src.fl.data_partition, src.fl.simulation
for _mod in [src.config, src.model, src.fl.data_partition, src.fl.simulation]:
    importlib.reload(_mod)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import LABEL_NAMES, MODELS_DIR
from src.fl.data_partition import partition_by_subject, summarize_partitions
from src.fl.simulation import FLRunConfig, run

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

## Inspect the per-client partitions
Each subject = one client. The scaler is fit per-client on each client's own training data.

In [ ]:
partitions = partition_by_subject()
summary = summarize_partitions(partitions)
# Normalise column names
summary = (summary
    .rename(columns={'baseline': 'normal'})
    .drop(columns=['amusement'], errors='ignore'))
summary

In [ ]:
class_cols = [c for c in summary.columns if c not in ('n_train', 'n_test')]
props = summary[class_cols].div(summary[class_cols].sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(props, annot=True, fmt='.2f', cmap='magma', cbar_kws={'label': 'class share'}, ax=ax)
ax.set_title('Per-client class distribution (proportions)')

## Run a small simulation (warm-started from the previous previously generated model)
We use a 4-subject subset for a fast test.

In [ ]:
cfg = FLRunConfig(
    rounds=3,
    local_epochs=2,
    batch_size=32,
    subjects=['S2', 'S3', 'S4', 'S5'],
    output=MODELS_DIR / 'fl_global_smoke.keras',
)
history, final_weights = run(cfg)

## Round-by-round aggregated metrics

In [ ]:
def history_to_df(pairs):
    return pd.DataFrame(pairs, columns=['round', 'value']).set_index('round')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

loss_df = history_to_df(history.losses_distributed)
loss_df.plot(ax=axes[0], marker='o', legend=False)
axes[0].set_title('Aggregated evaluation loss'); axes[0].set_ylabel('loss')

if 'accuracy' in history.metrics_distributed:
    acc_df = history_to_df(history.metrics_distributed['accuracy'])
    acc_df.plot(ax=axes[1], marker='o', color='green', legend=False)
    axes[1].set_title('Aggregated evaluation accuracy'); axes[1].set_ylabel('accuracy')
    axes[1].set_ylim(0, 1)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for k, series in history.metrics_distributed_fit.items():
    history_to_df(series).rename(columns={'value': k}).plot(ax=ax, marker='o')
ax.set_title('Fit metrics aggregated over clients (training side)')
ax.set_xlabel('round')

## Per-client final evaluation accuracy
We re-evaluate the final aggregated weights locally on each client's own held-out 20 % to see how the global model behaves on every user.

In [ ]:
import tensorflow as tf
from src.model import build_lstm

# Derive n_classes from the actual partition labels, not from LABEL_NAMES
# (avoids stale-kernel issues when the config changes)
n_classes = int(max(p.y_train.max() for p in partitions.values()) + 1)

global_model = build_lstm(
    input_shape=next(iter(partitions.values())).X_train.shape[1:],
    n_classes=n_classes,
)
global_model.set_weights(final_weights)

rows = []
for sid in cfg.subjects:
    p = partitions[sid]
    loss, acc = global_model.evaluate(p.X_test, p.y_test, verbose=0)
    rows.append({'subject': sid, 'loss': loss, 'accuracy': acc, 'n_test': p.n_test})
per_client = pd.DataFrame(rows).set_index('subject')
display(per_client.round(3))

fig, ax = plt.subplots(figsize=(6, 3))
per_client['accuracy'].plot(kind='bar', color='#3a86ff', ax=ax)
ax.set_ylim(0, 1); ax.set_title('Final global model — per-client test accuracy')
for c in ax.containers: ax.bar_label(c, fmt='%.2f')
    

Throughout the entire run, no raw signal ever crosses the network, only the models weights and scalar metrics